In [1]:
"""
Pneumothorax Detection using ResNet50V2
Following CheXNet methodology EXACTLY

Dataset: ChestX-ray14 (50K random sample)
Task: Binary classification (Pneumothorax vs. Not Pneumothorax)
Architecture: ResNet50V2 with ImageNet pretraining
"""

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


TensorFlow Version: 2.10.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================

CLASS_NAMES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'No Finding'
]

TARGET_DISEASE = 'Pneumothorax'

# Image parameters (CheXNet)
IMG_SIZE = 224
BATCH_SIZE = 16
CHANNELS = 3

# Training parameters (CheXNet)
INITIAL_LR = 0.001
EPOCHS = 50

# Class distribution
TOTAL_IMAGES = 50000
PNEUMOTHORAX_COUNT = 5297
NEGATIVE_COUNT = TOTAL_IMAGES - PNEUMOTHORAX_COUNT

# Calculate class weights (CheXNet formula)
WEIGHT_POS = NEGATIVE_COUNT / TOTAL_IMAGES
WEIGHT_NEG = PNEUMOTHORAX_COUNT / TOTAL_IMAGES

print(f"\n{'='*80}")
print(f"CLASS DISTRIBUTION")
print(f"{'='*80}")
print(f"Total: {TOTAL_IMAGES:,}")
print(f"Positive: {PNEUMOTHORAX_COUNT:,} ({PNEUMOTHORAX_COUNT/TOTAL_IMAGES*100:.2f}%)")
print(f"Negative: {NEGATIVE_COUNT:,} ({NEGATIVE_COUNT/TOTAL_IMAGES*100:.2f}%)")
print(f"Ratio: 1:{NEGATIVE_COUNT/PNEUMOTHORAX_COUNT:.1f}")
print(f"\nWeights: w+ = {WEIGHT_POS:.4f}, w- = {WEIGHT_NEG:.4f}")
print(f"{'='*80}\n")


CLASS DISTRIBUTION
Total: 50,000
Positive: 5,297 (10.59%)
Negative: 44,703 (89.41%)
Ratio: 1:8.4

Weights: w+ = 0.8941, w- = 0.1059



In [3]:
# ============================================================================
# DATA LOADING
# ============================================================================

BASE_DIR = "./dataset_unbalanced/"
CSV_PATH = os.path.join(BASE_DIR, "Data_Entry_2017.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

print(f"Loading dataset from: {CSV_PATH}")

df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset: {len(df_full):,} images")

# Parse labels
def parse_labels(finding_labels):
    labels = {class_name: 0 for class_name in CLASS_NAMES}
    if pd.isna(finding_labels) or finding_labels == "":
        labels["No Finding"] = 1
        return labels
    if finding_labels == "No Finding":
        labels["No Finding"] = 1
        return labels
    diseases = finding_labels.split("|")
    for disease in diseases:
        disease = disease.strip()
        if disease in CLASS_NAMES:
            labels[disease] = 1
    return labels

if 'Finding Labels' in df_full.columns:
    label_df = df_full['Finding Labels'].apply(parse_labels).apply(pd.Series)
    df_full = pd.concat([df_full, label_df], axis=1)
else:
    raise ValueError("Cannot find 'Finding Labels' column")

# Create image paths
def create_image_path(filename):
    return os.path.join(IMAGE_DIR, filename)

if 'Image Index' in df_full.columns:
    df_full['image_path'] = df_full['Image Index'].apply(create_image_path)
else:
    df_full['image_path'] = df_full.iloc[:, 0].apply(create_image_path)

# Verify images exist
valid_mask = df_full['image_path'].apply(os.path.exists)
df_full = df_full[valid_mask].reset_index(drop=True)
print(f"Valid images: {len(df_full):,}")

# Random sampling
SAMPLE_SIZE = 50000
if len(df_full) > SAMPLE_SIZE:
    df = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    df = df_full

# Binary labels
df['label'] = df[TARGET_DISEASE].astype(np.float32)

pos_count = df['label'].sum()
print(f"\nActual distribution: {int(pos_count):,} positive ({pos_count/len(df)*100:.2f}%)")

Loading dataset from: ./dataset_unbalanced/Data_Entry_2017.csv
Full dataset: 112,120 images
Valid images: 112,120

Actual distribution: 2,372 positive (4.74%)


In [4]:
# ============================================================================
# TRAIN/VAL/TEST SPLIT (70/10/20 by patients - CheXNet)
# ============================================================================

print(f"\n{'='*80}")
print("TRAIN/VAL/TEST SPLIT")
print(f"{'='*80}")

if 'Patient ID' in df.columns:
    unique_patients = df['Patient ID'].unique()
    train_patients, temp_patients = train_test_split(
        unique_patients, test_size=0.3, random_state=42
    )
    val_patients, test_patients = train_test_split(
        temp_patients, test_size=0.67, random_state=42
    )
    df_train = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    df_val = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    df_test = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)
else:
    df_train, temp = train_test_split(df, test_size=0.3, random_state=42)
    df_val, df_test = train_test_split(temp, test_size=0.67, random_state=42)

print(f"Train: {len(df_train):,} images")
print(f"Val:   {len(df_val):,} images")
print(f"Test:  {len(df_test):,} images")
print(f"{'='*80}\n")


TRAIN/VAL/TEST SPLIT
Train: 34,699 images
Val:   4,888 images
Test:  10,413 images



In [5]:
# ============================================================================
# TF.DATA PIPELINE (CheXNet preprocessing)
# ============================================================================

IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406])
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225])

def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - IMAGENET_MEAN) / IMAGENET_STD
    return image, label

def augment_image(image, label):
    """Random horizontal flipping only (CheXNet)"""
    image = tf.image.random_flip_left_right(image)
    return image, label

def create_dataset(dataframe, batch_size, shuffle=False, augment=False):
    image_paths = dataframe['image_path'].values
    labels = dataframe['label'].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = create_dataset(df_train, BATCH_SIZE, shuffle=True, augment=True)
val_dataset = create_dataset(df_val, BATCH_SIZE, shuffle=False, augment=False)
test_dataset = create_dataset(df_test, BATCH_SIZE, shuffle=False, augment=False)

print("Datasets created with tf.data pipeline")

Datasets created with tf.data pipeline


In [6]:
# ============================================================================
# MODEL - ResNet50V2 (following CheXNet architecture)
# ============================================================================

print(f"\n{'='*80}")
print("BUILDING MODEL")
print(f"{'='*80}")

def build_model():
    base_model = ResNet50V2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
        pooling='avg'
    )
    
    # End-to-end training (CheXNet trains all weights)
    base_model.trainable = True
    
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    x = base_model(inputs, training=True)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_model()
print(f"Model created: {model.count_params():,} parameters")



BUILDING MODEL
Model created: 23,566,849 parameters


In [7]:
# ============================================================================
# WEIGHTED BINARY CROSS-ENTROPY (CheXNet loss function)
# ============================================================================

def weighted_binary_crossentropy(y_true, y_pred):
    """CheXNet loss: L = -w+ * y * log(p) - w- * (1-y) * log(1-p)"""
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    loss = -(WEIGHT_POS * y_true * tf.math.log(y_pred) + 
             WEIGHT_NEG * (1 - y_true) * tf.math.log(1 - y_pred))
    return tf.reduce_mean(loss)

In [8]:
# ============================================================================
# COMPILE (CheXNet optimizer settings)
# ============================================================================

optimizer = keras.optimizers.Adam(learning_rate=INITIAL_LR, beta_1=0.9, beta_2=0.999)

model.compile(
    optimizer=optimizer,
    loss=weighted_binary_crossentropy,
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

print("Compiled with Adam (lr=0.001, β1=0.9, β2=0.999)")
print("Loss: Weighted Binary Cross-Entropy")

Compiled with Adam (lr=0.001, β1=0.9, β2=0.999)
Loss: Weighted Binary Cross-Entropy


In [9]:
# ============================================================================
# CALLBACKS (CheXNet settings)
# ============================================================================

callbacks = [
    ModelCheckpoint(
        'best_pneumothorax_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=3,
        verbose=1,
        min_lr=1e-7
    )
]

print("Callbacks: ModelCheckpoint, ReduceLROnPlateau")

Callbacks: ModelCheckpoint, ReduceLROnPlateau


In [10]:
# ============================================================================
# TRAINING
# ============================================================================

print(f"\n{'='*80}")
print("TRAINING")
print(f"{'='*80}\n")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


TRAINING

Epoch 1/50
2169/2169 [==============================] - ETA: 0s - loss: 0.0930 - accuracy: 0.9295 - auc: 0.5430 - precision: 0.0465 - recall: 0.0249
Epoch 1: val_loss improved from inf to 0.08544, saving model to best_pneumothorax_model.h5
2169/2169 [==============================] - 301s 134ms/step - loss: 0.0930 - accuracy: 0.9295 - auc: 0.5430 - precision: 0.0465 - recall: 0.0249 - val_loss: 0.0854 - val_accuracy: 0.9556 - val_auc: 0.5567 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - lr: 0.0010
Epoch 2/50
2169/2169 [==============================] - ETA: 0s - loss: 0.0882 - accuracy: 0.9442 - auc: 0.5656 - precision: 0.0439 - recall: 0.0085
Epoch 2: val_loss improved from 0.08544 to 0.08355, saving model to best_pneumothorax_model.h5
2169/2169 [==============================] - 287s 132ms/step - loss: 0.0882 - accuracy: 0.9442 - auc: 0.5656 - precision: 0.0439 - recall: 0.0085 - val_loss: 0.0835 - val_accuracy: 0.9556 - val_auc: 0.5575 - val_precision: 0.0000e+00

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# MODEL ARCHITECTURE - ResNet50V2
# ============================================================================

print("\n" + "="*80)
print("BUILDING MODEL: ResNet50V2")
print("="*80)

def build_model():
    """Build ResNet50V2 model following CheXNet architecture pattern"""
    
    # Load pretrained ResNet50V2 (ImageNet weights)
    base_model = ResNet50V2(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
        pooling='avg'  # Global average pooling
    )
    
    # Unfreeze the base model for end-to-end training (like CheXNet)
    base_model.trainable = True
    
    # Create model
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    x = base_model(inputs, training=True)  # Enable training mode
    
    # Single output with sigmoid (binary classification)
    outputs = layers.Dense(1, activation='sigmoid', name='pneumothorax_output')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='ResNet50V2_Pneumothorax')
    
    return model

model = build_model()

print(f"Model created successfully")
print(f"Total parameters: {model.count_params():,}")
print(f"Base model: ResNet50V2 (pretrained on ImageNet)")
print(f"Output: Single sigmoid neuron for binary classification")

In [ ]:
# ============================================================================
# VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(history.history['loss'], label='Train')
axes[0, 0].plot(history.history['val_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history.history['auc'], label='Train')
axes[0, 1].plot(history.history['val_auc'], label='Val')
axes[0, 1].set_title('AUC')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history.history['precision'], label='Train')
axes[1, 0].plot(history.history['val_precision'], label='Val')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history.history['recall'], label='Train')
axes[1, 1].plot(history.history['val_recall'], label='Val')
axes[1, 1].set_title('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300)
print("\nSaved: training_history.png")

print(f"\n{'='*80}")
print("COMPLETE")
print(f"{'='*80}")